# Load Inventory Sets from Rebrickable CSV Download
Downloads `inventory_sets.csv.gz` from the Rebrickable bulk download page and loads it into a Spark DataFrame.

## Imports and Configuration

Import standard and third-party libraries needed for HTTP download, gzip decompression, Pandas/Spark processing, and Delta Lake operations. `DOWNLOAD_URL` points to the Rebrickable bulk-download endpoint for the inventory sets dataset.

In [ ]:
import gzip
import io
import requests
from datetime import datetime

import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

DOWNLOAD_URL = "https://cdn.rebrickable.com/media/downloads/inventory_sets.csv.gz"

## Download and Inspect Source Data

Fetch the gzip-compressed CSV directly into memory via HTTP, decompress it, and load it into a Pandas DataFrame. The row count and column dtypes are printed so you can verify the download before proceeding.

In [ ]:
response = requests.get(DOWNLOAD_URL, timeout=60)
response.raise_for_status()

with gzip.open(io.BytesIO(response.content)) as f:
    df_pd = pd.read_csv(f)

print(f"Downloaded {len(df_pd)} rows")
print(df_pd.dtypes)
df_pd.head()

## Convert to Spark DataFrame

Define an explicit schema (`inventory_id`, `set_num`, `quantity`) and convert the Pandas DataFrame to a typed Spark DataFrame. Printing the schema and previewing 10 rows confirms the conversion looks correct before writing.

In [ ]:
schema = StructType([
    StructField("inventory_id", IntegerType(), nullable=False),
    StructField("set_num",      StringType(),  nullable=False),
    StructField("quantity",     IntegerType(), nullable=False),
])

rows = [
    (
        int(row["inventory_id"]),
        str(row["set_num"]),
        int(row["quantity"]),
    )
    for _, row in df_pd.iterrows()
]

spark = SparkSession.builder.getOrCreate()
df_inventory_sets = spark.createDataFrame(rows, schema=schema)

df_inventory_sets.printSchema()
df_inventory_sets.display(10, truncate=False)

## Write Raw Parquet to External Volume

Append audit columns (`_load_datetime`, `_record_source`) and write the data as a single Parquet file to a date/time-partitioned path under the Unity Catalog external volume. The `part-` file is renamed to a deterministic filename and the temporary staging directory is removed.

In [ ]:
# Generate timestamp components for partitioned path and filename
now   = datetime.utcnow()
year  = now.strftime("%Y")
month = now.strftime("%m")
day   = now.strftime("%d")
ts    = now.strftime("%Y%m%d_%H%M%S")

# Add audit columns
df_inventory_sets_out = (
    df_inventory_sets
    .withColumn("_load_datetime", current_timestamp())
    .withColumn("_record_source", lit("CSV_DOWNLOAD"))
)

# Update the base path below to match your Unity Catalog external volume mount point
EXTERNAL_VOLUME_BASE = "/Volumes/lego_brickbase/bronze/raw_data_volume/inventory_sets"
PARTITION_PATH       = f"{EXTERNAL_VOLUME_BASE}/{year}/{month}/{day}/{ts}"
TEMP_PATH            = f"{PARTITION_PATH}/_tmp"
FILE_NAME            = f"raw_inventory_sets_{ts}.parquet"
FINAL_PATH           = f"{PARTITION_PATH}/{FILE_NAME}"

# Write as a single Parquet file to a temp staging directory
df_inventory_sets_out.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet(TEMP_PATH)

# Locate the single part file and rename it to the desired filename
part_files = [f.path for f in dbutils.fs.ls(TEMP_PATH) if f.name.startswith("part-")]
dbutils.fs.mv(part_files[0], FINAL_PATH)
dbutils.fs.rm(TEMP_PATH, recurse=True)

print(f"Inventory sets written to: {FINAL_PATH}")

## Merge into SCD2 Delta Table

Implement a Type 2 Slowly Changing Dimension (SCD2) merge against the Delta table:

1. **First load** — all incoming rows are written as current records.
2. **Subsequent loads** — a two-step merge is applied:
   - *Step 1:* Expire changed records (`valid_to` = now, `is_current` = false) and soft-delete rows no longer present in the source (`is_deleted` = true).
   - *Step 2:* Insert new records and new versions of changed records as current rows.

Finally, the Delta table is registered in Unity Catalog under `lego_brickbase.bronze.raw_inventory_sets` if it does not already exist.

In [ ]:
DELTA_TABLE_PATH  = "/Volumes/lego_brickbase/bronze/delta_volume/inventory_sets"
CATALOG_TABLE     = "lego_brickbase.bronze.raw_inventory_sets"

# Natural key and tracked business columns
KEY_COLS     = ["inventory_id", "set_num"]
TRACKED_COLS = ["quantity"]

# Read the Parquet file written in the previous step
df_source = spark.read.parquet(FINAL_PATH)

# Attach SCD2 metadata columns
df_incoming = (
    df_source
    .withColumn("valid_from", current_timestamp())
    .withColumn("valid_to",   lit(None).cast(TimestampType()))
    .withColumn("is_current", lit(True))
    .withColumn("is_deleted", lit(False))
)

if not DeltaTable.isDeltaTable(spark, DELTA_TABLE_PATH):
    # First load — write all records as current
    df_incoming.write \
        .format("delta") \
        .mode("overwrite") \
        .save(DELTA_TABLE_PATH)
    print(f"Delta table created at {DELTA_TABLE_PATH}")
else:
    delta_table    = DeltaTable.forPath(spark, DELTA_TABLE_PATH)
    join_condition = " AND ".join([f"tgt.{c} = src.{c}" for c in KEY_COLS])
    change_cond    = " OR ".join([f"tgt.{c} != src.{c}" for c in TRACKED_COLS])

    # Step 1: Expire changed records; soft-delete records missing from source
    (
        delta_table.alias("tgt")
        .merge(df_incoming.alias("src"), f"{join_condition} AND tgt.is_current = true")
        .whenMatchedUpdate(
            condition=change_cond,
            set={"valid_to": "src.valid_from", "is_current": "false"}
        )
        .whenNotMatchedBySourceUpdate(
            condition="tgt.is_current = true",
            set={"valid_to": "current_timestamp()", "is_current": "false", "is_deleted": "true"}
        )
        .execute()
    )

    # Step 2: Insert new records and new versions of changed records
    df_current   = delta_table.toDF().filter("is_current = true")
    df_to_insert = df_incoming.join(df_current.select(*KEY_COLS), on=KEY_COLS, how="left_anti")
    df_to_insert.write.format("delta").mode("append").save(DELTA_TABLE_PATH)

row_count = DeltaTable.forPath(spark, DELTA_TABLE_PATH).toDF().filter("is_current = true").count()
print(f"Active rows in Delta table: {row_count:,}")

# Register the Delta table in Unity Catalog
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
    AS SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")
print(f"Catalog table ready: {CATALOG_TABLE}")